# Fixify ? Documentaci?n del front-end, versi?n 2

**Fecha: 02-Oct-2026.** Esta versi?n documenta el front-end refactorizado, con especial atenci?n a `services/api_client.py`, httpx y asyncio. La versi?n 1 se conserva como referencia hist?rica.

La aplicaci?n es de escritorio: Tkinter/ttkbootstrap dibuja ventanas; FastAPI recibe peticiones; PostgreSQL en Supabase guarda datos. El frontend no se conecta directamente a la BD.

## C?mo leer y ejecutar

1. Para entender la API, leer las secciones 1?9 en orden.
2. Para entender ventanas y asincron?a, continuar con las secciones 10?13.
3. Para comprobar el comportamiento, ejecutar los ejemplos simulados de la secci?n 14.
4. Para localizar una clase/m?todo, consultar el inventario y el c?digo de referencia de la secci?n 15.

Las celdas ejecutables usan exclusivamente `httpx.MockTransport`: **no env?an tr?fico de red, no abren ventanas ni leen credenciales**. Los fragmentos de la aplicaci?n est?n en Markdown, por lo que Run All no ejecuta la GUI. Seleccionar un kernel con las dependencias de `frontend/requirements.txt`. No se necesita instalar un paquete llamado asyncio: pertenece a Python.

La revisi?n y pruebas locales no acreditan conectividad real con Supabase ni aceptaci?n visual completa.

## 1. Qu? responsabilidad tiene cada pieza

```text
Ventana / vista (Tk, hilo principal)
  lee campos y pide una operaci?n
       ?
       ?
AsyncRunner.submit(coroutine, owner, on_success, on_error)
       ? agenda trabajo en el loop de un hilo separado
       ?
Service de dominio (Auth / Clients / Equipment / Users)
  decide ruta, verbo y datos
       ? await
       ?
ApiClient (nuestro adaptador com?n)
  agrega JWT, administra HTTP y transforma errores
       ? await
       ?
httpx.AsyncClient (objeto de la biblioteca httpx)
  transporta la solicitud y obtiene una Response
       ?
       ?
FastAPI ? repositorio SQLAlchemy ? PostgreSQL/Supabase
       ? respuesta
       ?
cola Python ? root.after ? callback de la vista (hilo Tk)
```

`ApiClient` y `httpx.AsyncClient` son clases diferentes. La primera es c?digo de Fixify; la segunda es la biblioteca que realiza HTTP. `ModuleService` comparte transporte y validaci?n de listas; cada service hereda de esa base. `Services` construye las instancias y las entrega a las vistas: no realiza solicitudes por s? mismo.

Se comparten objetos, no se copian: `services.clients.api is services.api` y `services.equipment.api is services.api` son verdaderos. Cambiar el token del transporte afecta a las pr?ximas solicitudes de todos los m?dulos.

## 2. Vocabulario m?nimo de HTTP y Python

| Concepto | Significado en Fixify | Ejemplo |
|---|---|---|
| URL base | Direcci?n del backend | `http://127.0.0.1:8000` |
| Ruta | Recurso relativo | `/equipos/` |
| GET | Consulta sin alta | Listar equipos |
| POST | Env?o para crear o autenticar | Registrar equipo / login |
| PATCH | Cambio parcial | Cambiar rol |
| Headers | Metadatos HTTP | Authorization |
| Body | Datos del env?o | JSON con marca/modelo |
| Query params | Filtros en URL | `/clientes/buscar?q=Ana` |
| Response | Objeto recibido de httpx | Status, headers y contenido |
| `dict` / `list` | Datos Python despu?s de decodificar JSON | Una entidad / varias entidades |
| `self` | La instancia actual de una clase | `self.token` pertenece al transporte |
| `_nombre` | Convenci?n de miembro interno | `_client` no es un service de dominio |
| Coroutine | Operaci?n async pendiente | `service.listar()` |
| Callback | Funci?n que se llama al terminar | `_loaded(records)` |

Una petici?n es un viaje al servidor. Una coroutine es una operaci?n pendiente de Python. Crear una coroutine no ejecuta todav?a ese viaje. El callback es la funci?n que procesa su resultado; no es la respuesta HTTP.

## 3. ApiClient: estado, constructor y ciclo de vida

Archivo: `frontend/src/services/api_client.py`.

| Atributo | Valor inicial | Uso |
|---|---|---|
| `base_url` | Argumento, variable FIXIFY_API_URL o direcci?n local | Localizar backend |
| `token` | None | JWT compartido de la sesi?n |
| `_transport` | Argumento opcional | Transporte real por defecto o simulador de pruebas |
| `_client` | None | AsyncClient creado en la primera solicitud |

`base_url or os.getenv(...)` elige una URL expl?cita si es verdadera; si se omite o est? vac?a, consulta el entorno. La URL no contiene las credenciales de Supabase. El argumento `transport` permite conectar el mismo c?digo a un simulador; no representa una tabla de BD.

El constructor solo guarda configuraci?n. La primera llamada a `request` crea `_client`; las siguientes lo reutilizan. Se evita abrir un cliente por cada registro o cada clic. Su creaci?n y uso ocurren en el loop del trabajador, el mismo donde se cierran las conexiones.

```text
ApiClient creado ? _client=None
primera request ? se crea AsyncClient
siguientes request ? mismo AsyncClient
aclose ? se cierran conexiones ? _client=None
```

`set_token(token)` no llama al servidor: cambia memoria local. `set_token(None)` elimina la credencial de futuras peticiones, pero no revoca un JWT ya emitido en el backend.

## 4. La firma de request, argumento por argumento

```python
async def request(self, method, path, *, authenticated=True, **kwargs):
```

| Elemento | Qu? significa | C?mo se utiliza |
|---|---|---|
| `async def` | Define una coroutine | Se espera con await o se agenda en AsyncRunner |
| `self` | ApiClient compartido | Da acceso a URL, token y conexiones |
| `method` | Verbo HTTP | GET, POST, PATCH; httpx tambi?n admite otros |
| `path` | Ruta relativa | `/clientes/` |
| `*` | A partir de aqu? los argumentos son por nombre | `authenticated=False` |
| `authenticated=True` | Habilita agregar JWT si hay uno | Login lo desactiva |
| `**kwargs` | Re?ne argumentos adicionales en un diccionario | json, data, params |

Ejemplo de expansi?n:

```python
await api.request('POST', '/clientes/', json={'nombre': 'Ana', 'telefono': '3312345678'})
```

Dentro de `request`, `kwargs` contiene `{'json': {...}}`. Luego `**kwargs` en la llamada a httpx vuelve a expandirlo como argumentos por nombre. No se modifica el payload por magia: el service lo construye expl?citamente.

`authenticated=True` **no valida** una cuenta ni garantiza que exista token. Si token es None se env?an headers vac?os y el servidor decide si rechaza. No se deben pasar `headers` dentro de kwargs con el dise?o actual: `request` ya pasa ese argumento, y repetirlo causar?a TypeError. Cambios futuros de headers deben centralizarse en el transporte.

## 5. request por etapas: desde el clic hasta los datos

### 5.1 Preparar el cliente

`if self._client is None` crea `httpx.AsyncClient(base_url=..., timeout=5, transport=..., follow_redirects=True)`. El proyecto configura un timeout de cinco segundos y sigue redirecciones. Ese timeout controla esperas de operaciones HTTP; no debe interpretarse como un cron?metro total exacto para todo el flujo de UI. [Referencia de timeout HTTPX](https://www.python-httpx.org/advanced/timeouts/).

### 5.2 Preparar autorizaci?n

```python
headers = {'Authorization': f'Bearer {self.token}'} if authenticated and self.token else {}
```

Solo se agrega la cabecera cuando ambos valores lo permiten. Bearer anuncia al backend el esquema de autenticaci?n. No imprimir token, contrase?a ni cabeceras reales en logs/notebooks.

### 5.3 Esperar la red

```python
response = await self._client.request(method, path, headers=headers, **kwargs)
```

Mientras la coroutine espera una operaci?n de E/S, el loop puede atender otras tareas. En Fixify ese loop est? en el hilo de trabajo; Tk sigue procesando eventos en su hilo. El archivo no mueve widgets ni llama a messagebox.

### 5.4 Interpretar respuesta

`response.is_error` identifica errores HTTP y hace que el adaptador lance ApiError. Si la respuesta es 204 se devuelve None: no hay contenido para decodificar. En otro caso `response.json()` retorna datos Python. La aplicaci?n no devuelve el objeto Response a las vistas, sino el contenido decodificado.

Que la respuesta sea 200 no garantiza que su JSON tenga el contrato correcto. `ModuleService._list` comprueba lista y valida cada registro con su modelo; `AuthService.login` comprueba token y rol. Cada entidad se valida mediante un contrato Pydantic antes de entregarla a la vista.

## 6. JSON, formulario y filtros: no son intercambiables

| Argumento | Uso actual | Ejemplo del m?dulo |
|---|---|---|
| `json={...}` | Cuerpo JSON | Alta cliente/equipo/usuario y cambio de rol |
| `data={...}` | Formulario codificado | POST /auth/login |
| `params={...}` | Filtros en la URL | Buscar cliente por q |

`AuthService.login` usa formulario porque el backend recibe OAuth2PasswordRequestForm. Si se cambia a JSON sin cambiar el backend, el contrato no coincide. `ClientsService.buscar` deja a httpx codificar el t?rmino: no concatena texto manualmente en la URL. Los opcionales de alta se convierten de cadena vac?a a None, que se serializa como null.

Ejemplos del c?digo local:

```python
await api.request('POST', '/auth/login', authenticated=False,
                  data={'username': username, 'password': password})
await api.request('POST', '/equipos/', json=payload)
await api.request('GET', '/clientes/buscar', params={'q': termino})
```

La configuraci?n de par?metros y cuerpos se apoya en la [gu?a de HTTPX](https://www.python-httpx.org/quickstart/); las rutas y conversiones anteriores provienen de los services de Fixify.

## 7. ApiError: errores de transporte, HTTP y contrato

`ApiError` hereda de Exception, conserva un mensaje legible y un `status_code` opcional. `super().__init__(message)` permite que `str(error)` sea el mensaje mostrado por la UI.

| Situaci?n | Tratamiento local | status_code |
|---|---|---|
| TimeoutException | ?El servidor tard? demasiado?? | None |
| RequestError | ?No se pudo conectar?? | None |
| HTTP 401/403/404/422/500 | `_error_message` extrae detail o genera mensaje HTTP | C?digo recibido |
| Respuesta no JSON cuando se espera JSON | ?Respuesta inv?lida? | None |
| Lista o login con estructura incorrecta | El service/base lanza ApiError | None |

Timeout se captura antes de RequestError para conservar un mensaje espec?fico. `raise ... from exc` mantiene la causa para depuraci?n, mientras la UI utiliza el texto simple del nuevo error.

`_error_message` es staticmethod: no utiliza token ni estado de la instancia, solo la respuesta recibida. Si detail es lista de errores Pydantic, une sus mensajes con saltos de l?nea; si es cadena, la conserva. Si no entiende la respuesta, genera `La solicitud fall? (HTTP N).` No existe reintento autom?tico.

Cancelaci?n no se interpreta como ?xito ni como fallo HTTP. AsyncRunner descarta futures cancelados. Si el servidor ya confirm? una escritura, cancelar la espera no la deshace: antes de repetir un alta, consultar si se guard?.

## 8. ModuleService, herencia e inyecci?n

```python
class EquipmentService(ModuleService):
    async def listar(self):
        return await self._list('/equipos/', Equipo)
```

EquipmentService recibe el constructor de ModuleService por herencia. `ModuleService.__init__(api)` conserva `self.api`; `_list(path, **kwargs)` hace GET, recibe JSON, exige una lista de diccionarios y convierte cada elemento al modelo indicado. Un listado vac?o `[]` es v?lido.

`Services(api=None)` usa el ApiClient recibido o crea uno. Construye cuatro services que apuntan al mismo transporte. Las vistas reciben `services` y `runner` en sus constructores. Esa inyecci?n permite sustituir solo el transporte en pruebas y conservar los services reales.

| Clase / archivo | Operaciones | Transformaci?n de entrada |
|---|---|---|
| AuthService / auth_service.py | login | Formulario; valida respuesta; no guarda JWT |
| ClientsService / clients_service.py | listar, crear, buscar | Opcionales a None; q en params |
| EquipmentService / equipment_service.py | listar, crear | client_id a int; serie vac?a a None |
| UsersService / users_service.py | listar, crear, cambiar_rol, activar | PATCH parcial; elige habilitar/inhabilitar |

Los services no contienen controles de Tk, no deciden colores, no leen Entry y no abren ventanas. Tampoco son controles de seguridad suficientes: el backend vuelve a autorizar cada operaci?n.

## 9. Recorrido de equipos y relaci?n con el diagrama de BD

### Consultar

1. CollectionView.load pide la coroutine EquipmentView.fetch.
2. EquipmentView llama EquipmentService.listar.
3. ModuleService._list llama ApiClient.request GET `/equipos/`.
4. httpx obtiene el JSON del router; se valida lista y se obtiene una lista de modelos Equipo.
5. AsyncRunner entrega la lista a CollectionView._loaded en Tk.
6. EquipmentView.row transforma cada registro a columnas de tabla.

### Registrar

1. El bot?n Nuevo consulta clientes sin bloquear la UI.
2. La vista crea un mapa etiqueta visible ? ID y un formulario.
3. El usuario completa campos; FormDialog lee valores en el hilo Tk.
4. La coroutine de guardado convierte la etiqueta de propietario al ID.
5. EquipmentService construye payload JSON y llama POST `/equipos/`.
6. El backend comprueba cliente, inserta y responde 201 con folio/detalles.
7. El callback informa alta y vuelve a cargar el listado.

El backend sigue router ? repositorio ? modelo/esquema. La implementaci?n de equipos ahora reconoce los nombres del diagrama entregado:

| API / atributo Python | Columna f?sica SQL |
|---|---|
| id | id_equipo |
| client_id | id_cliente |
| problema_reportado | falla_reportada |
| tipo, marca, modelo, numero_serie | Mismo nombre |

La FK apunta a `clientes.id_cliente`. El GET incluye numero_serie, incluso null cuando no existe. El orden del listado es ascendente por ID. Tanto POST como GET exigen admin/recepcion. `EQ-001` identifica equipo, no una orden de servicio.

El cambio es de mapeo ORM y endpoint; no ejecuta renombrados ni migraciones en Supabase. El diagrama no sustituye inspeccionar el esquema remoto. Otros m?dulos conservan diferencias previas, por ejemplo la columna direccion de clientes y el dise?o users frente a usuarios/roles.

## 10. async, await y el puente con Tk

`async def listar` produce una coroutine al llamarse. `await` espera su resultado dentro de otra coroutine. Tk espera funciones normales en command; por ello un bot?n no debe recibir directamente una funci?n async.

```python
# Callback normal de Tk:
self.runner.submit(self.services.clients.listar(), self, self._loaded, self._failed)
```

No se usa `asyncio.run` dentro de un callback: ese m?todo esperar?a all? a que termine el flujo. En notebook se puede usar await directamente. En un script independiente sin GUI, asyncio.run puede ser adecuado como entrada ?nica.

### AsyncRunner por m?todo

| M?todo | Trabajo |
|---|---|
| __init__(root) | Crea loop, cola, registro de futures y trabajador; programa after |
| _run() | Instala el loop en el hilo y lo mantiene activo; cierra async generators al salir |
| submit(coroutine, owner, on_success, on_error) | Agenda coroutine y registra due?o/callbacks |
| cancel_owner(owner) | Elimina callbacks y solicita cancelaci?n de las tareas de ese due?o |
| _poll() | En el hilo Tk, toma futures terminados de la cola y llama al callback correspondiente |
| close(cleanup, on_closed) | Inicia cancelaci?n y cierre HTTP sin bloquear Tk |
| _wait_thread(on_closed) | Espera con after hasta que el hilo termina y cierra la ventana |

`run_coroutine_threadsafe` devuelve un Future de concurrent.futures para coordinar hilos. Su done callback solo pone el Future en Queue; no llama a Tk. `_poll` obtiene result() despu?s de finalizar el Future, comprueba que la vista exista y procesa resultado/error. El polling ocurre aproximadamente cada 30 ms; no significa que haya una llamada de red cada 30 ms. [Referencia de asyncio](https://docs.python.org/3/library/asyncio-task.html#asyncio.run_coroutine_threadsafe).

## 11. Concurrencia, cancelaci?n y cierre

El dashboard realiza consultas independientes con asyncio.gather y return_exceptions=True. Los resultados conservan el orden de las consultas; un fallo se representa como excepci?n en esa posici?n y no pierde las m?tricas que s? cargaron. Usuarios solo se consulta para admin; t?cnico no intenta leer endpoints prohibidos de clientes/equipos.

Cerrar una vista cancela las tareas de su owner. Cerrar el formulario hace lo mismo para su guardado. CollectionView.load cancela su lectura anterior antes de pedir una nueva; as? reduce recargas desactualizadas. FormDialog conserva una ventana por m?dulo, y botones deshabilitados evitan repetir clics durante la espera.

`AsyncRunner.close` deja de programar polling, cancela tareas, espera cancelaciones, ejecuta `api.aclose()` en el loop de trabajo, detiene el loop y comprueba que el hilo termin? antes de destruir la ventana principal. `aclose` libera HTTP y deja `_client=None`. [Referencia del ciclo de vida AsyncClient](https://www.python-httpx.org/async/).

No todas las esperas son cancelaci?n de negocio: el backend puede haber guardado antes de recibir la desconexi?n. No hay WebSocket, suscripci?n Supabase ni actualizaci?n autom?tica desde eventos remotos: la reactividad actual mantiene la UI disponible durante HTTP y actualiza datos cuando llegan. Otros cambios externos requieren Actualizar o nueva consulta.

## 12. Componentes de ttkbootstrap y vistas

| Componente | Reutilizaci?n | Configuraci?n importante |
|---|---|---|
| Field | Definici?n de campo sin widgets | name, label, required, secret, choices |
| create_table | Treeview con encabezados y scroll vertical | columns y height |
| AsyncFrame | Cancelaci?n por due?o al destruir | services, runner, role |
| FormDialog | Entradas/selectores y guardado | fields, save(values), on_saved(result) |
| CollectionView | Listado/recarga/alta y estado de solicitud | title, columns, fields; fetch, save, row |
| ClientsView | Campos y transformaci?n de cliente | Opcionales con guion visible |
| EquipmentView | Consulta de propietarios y formulario de equipos | Selector etiqueta?ID |
| UsersView | Alta y cambios de acceso | Mapa etiquetas?roles API y usuario seleccionado |
| DashboardView | M?tricas concurrentes y resumen | Consultas seg?n rol |

Field es dataclass frozen: su definici?n es inmutable, no almacena valores editados. FormDialog captura valores desde controles; aplica strip a datos normales pero preserva contrase?as. La validaci?n de presencia en cliente mejora feedback; backend valida formatos, longitudes e integridad.

CollectionView es una base de reutilizaci?n: una subclase implementa fetch (consulta), save (alta) y row (registro?valores). Las funciones base lanzan NotImplementedError si falta implementaci?n. Usuarios traduce roles visibles a admin/tecnico/recepcion, pide confirmaci?n y recarga despu?s de cambiar acceso.

## 13. Sesi?n, navegaci?n y mantenimiento

`app/main.py` prepara imports desde src y crea VentanaLogin. Login lee credenciales, deshabilita bot?n, muestra Conectando y env?a trabajo al runner. AuthService valida respuesta, pero **no asigna token**: el callback de login lo hace si sigue activo. Luego abre SistemaGestion y oculta login.

SistemaGestion recibe rol de dominio, no etiqueta traducida, y decide men?s. Reemplazar vista destruye controles y tareas anteriores. Logout limpia JWT local, destruye gesti?n y vuelve a login. Cerrar aplicaci?n usa el cierre coordinado de recursos. Recuperaci?n de cuenta informa que se contacte al administrador: no existe env?o autom?tico.

### C?mo agregar o cambiar una operaci?n

- Cambiar ruta/payload en el service del dominio; conservar manejo HTTP en ApiClient.
- Para nuevos filtros usar params; para cuerpos usar el contrato acordado json/data.
- Si cambia la estructura de salida, ajustar esquema backend, validaci?n y row de la vista.
- Para un m?dulo de listado/alta, crear una subclase de CollectionView y registrarla en navegaci?n.
- Actualizar pruebas de contratos y esta documentaci?n cuando cambie comportamiento.

`models/database.py` conserva un prototipo de usuarios en memoria; no participa en login actual. No se reproducen sus contrase?as en este notebook.

### L?mite de esta implementaci?n

No hay refresh/revocaci?n de JWT, pol?tica de retries, cache o paginaci?n. Los permisos se basan tambi?n en backend y sus dependencias actuales mantienen rol del JWT emitido. Estas funciones no se atribuyen a httpx por el hecho de usar AsyncClient.

## 14. Laboratorio seguro: httpx con respuestas simuladas

Las siguientes celdas importan **los services reales** y usan MockTransport. Los datos son sint?ticos; el token de demostraci?n nunca se imprime. Ejecutar celdas en orden desde el repositorio o su carpeta docs. Cada ejemplo cierra su cliente en finally.

El handler simulado hace el papel del servidor y permite inspeccionar JSON, formulario y headers sin una conexi?n externa.

In [1]:
from pathlib import Path
import sys
import asyncio
import json
import httpx

raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'frontend/src/services/api_client.py').is_file()), None)
if raiz is None:
    raise RuntimeError('Abra el notebook dentro del repositorio Fixify.')
sys.path.insert(0, str(raiz / 'frontend/src'))
from services import Services
from services.api_client import ApiClient, ApiError
print('Services importados. No se abri? GUI ni conexi?n de red.')

Services importados. No se abri? GUI ni conexi?n de red.


In [2]:
async def ejemplo_alta_y_listado():
    registros = []
    solicitudes = []

    def servidor(request):
        # Observamos presencia de JWT, no su contenido.
        solicitudes.append((request.method, request.url.path,
                             'authorization' in request.headers))
        if request.url.path == '/auth/login':
            assert request.headers['content-type'].startswith('application/x-www-form-urlencoded')
            return httpx.Response(200, json={'access_token': 'token-sintetico', 'token_type': 'bearer', 'role': 'recepcion'})
        if request.method == 'POST':
            entidad = {'id': len(registros) + 1, **json.loads(request.content)}
            registros.append(entidad)
            return httpx.Response(201, json={'folio': f"EQ-{entidad['id']:03d}", 'mensaje': 'Guardado', 'detalles': {
                'id_interno': entidad['id'], 'tipo': entidad['tipo'], 'marca': entidad['marca'],
                'modelo': entidad['modelo'], 'cliente_id': entidad['client_id']}})
        return httpx.Response(200, json=registros)

    api = ApiClient(transport=httpx.MockTransport(servidor))
    services = Services(api)
    try:
        sesion = await services.auth.login('demo', 'clave-sintetica')
        api.set_token(sesion.access_token)
        resultado = await services.equipment.crear(
            tipo='Laptop', marca='Marca', modelo='Modelo',
            problema_reportado='No enciende', client_id=7, numero_serie='DEMO-001')
        equipos = await services.equipment.listar()
        assert equipos[0].numero_serie == 'DEMO-001'
        print('Folio:', resultado.folio)
        print('Equipos consultados:', len(equipos))
        print('Solicitudes (m?todo, ruta, tiene autorizaci?n):', solicitudes)
        print('Transporte compartido:', services.clients.api is services.equipment.api)
    finally:
        await api.aclose()
    assert api._client is None
    print('HTTP cerrado.')

await ejemplo_alta_y_listado()

Folio: EQ-001
Equipos consultados: 1
Solicitudes (m?todo, ruta, tiene autorizaci?n): [('POST', '/auth/login', False), ('POST', '/equipos/', True), ('GET', '/equipos/', True)]
Transporte compartido: True
HTTP cerrado.


In [3]:
async def ejemplo_errores():
    def servidor(request):
        return httpx.Response(422, json={'detail': [{'msg': 'Nombre obligatorio'}]})

    api = ApiClient(transport=httpx.MockTransport(servidor))
    try:
        await api.request('POST', '/clientes/', json={})
    except ApiError as error:
        print('Mensaje visible:', str(error))
        print('C?digo HTTP:', error.status_code)
        assert error.status_code == 422
    finally:
        await api.aclose()

await ejemplo_errores()

Mensaje visible: Nombre obligatorio
C?digo HTTP: 422


In [4]:
async def ejemplo_concurrencia():
    iniciadas = set()
    todas_iniciadas = asyncio.Event()

    async def servidor(request):
        iniciadas.add(request.url.path)
        if len(iniciadas) == 2:
            todas_iniciadas.set()
        # Ambos GET deben comenzar antes de que cualquiera finalice.
        await asyncio.wait_for(todas_iniciadas.wait(), timeout=1)
        return httpx.Response(200, json=[])

    api = ApiClient(transport=httpx.MockTransport(servidor))
    services = Services(api)
    try:
        clientes, equipos = await asyncio.gather(
            services.clients.listar(), services.equipment.listar())
        assert len(iniciadas) == 2
        print('Consultas concurrentes:', sorted(iniciadas))
        print('Resultados:', clientes, equipos)
    finally:
        await api.aclose()

await ejemplo_concurrencia()

Consultas concurrentes: ['/clientes/', '/equipos/']
Resultados: [] []


## Contrato de modelos ? actualizaci?n del 02-Oct-2026

Los services ya no retornan diccionarios de dominio. Retornan modelos Pydantic con atributos validados. El transporte ApiClient sigue retornando JSON decodificado; ModuleService._parse aplica el contrato y transforma ValidationError en ApiError sin mostrar los datos inv?lidos.

```text
Formulario ? modelo Crear/Login ? model_dump() ? HTTP
HTTP ? JSON ? modelo de respuesta ? atributos en la vista
```

| M?dulo | Entradas | Consultas | Confirmaciones |
|---|---|---|---|
| auth | Login | Sesion | Sesion con token_type bearer |
| clients | ClienteCrear | Cliente | ClienteRegistrado: mensaje + cliente |
| equipment | EquipoCrear | Equipo | EquipoRegistrado: mensaje + folio + DetallesEquipo |
| users | UsuarioCrear, RolActualizar | Usuario | UsuarioRegistrado o Mensaje |

Ejemplo de uso:

```python
equipos = await services.equipment.listar()
print(equipos[0].numero_serie)
sesion = await services.auth.login(username, password)
api.set_token(sesion.access_token)
```

`Contrato` hereda de BaseModel y configura strict=True, frozen=True y extra='ignore': valida tipos sin convertir cadenas a enteros/booleanos, evita modificaciones accidentales y descarta campos no declarados. Un ID debe ser entero positivo: enviar 7, no '7'. Los opcionales aceptan null/None o ausencia y tienen default None. Los roles est?n restringidos a admin, tecnico y recepcion con Literal.

Los modelos de alta incluyen las longitudes existentes del backend; correo se declara como string opcional y su formato lo valida el servidor. La autenticaci?n, unicidad e integridad siguen correspondiendo al backend. Ignorar campos adicionales permite evolucionar respuestas sin conservar hashes de usuarios; no soluciona que un backend los exponga en la red.

Pydantic se usa de forma declarativa: campos, tipos, Field y dos operaciones (`model_validate` / `model_dump`). No hay validadores personalizados, clases ORM ni conexi?n de BD en frontend. La representaci?n de password/access_token se oculta con repr=False; model_dump a?n contiene sus valores para enviar credenciales, por lo que no debe imprimirse en logs reales.

Las confirmaciones de alta tambi?n se validan: un HTTP 201 con JSON incompleto genera error de contrato y no se muestra como ?xito confirmado. Puede haber una escritura ya guardada en servidor: consultar antes de repetir un alta.

Referencias: [modelos Pydantic](https://docs.pydantic.dev/latest/concepts/models/) y [modo estricto](https://docs.pydantic.dev/latest/concepts/strict_mode/).


## 15. Inventario de archivos y c?digo de referencia vigente

Cada archivo incluye prop?sito, clases/m?todos con l?neas y docstrings, imports y una instant?nea de lectura. SHA-256 identifica la versi?n original del archivo. Cambios posteriores requieren actualizar la instant?nea; los ejemplos de laboratorio importan el c?digo actual.

Las explicaciones anteriores desarrollan la arquitectura; este inventario permite localizar cada parte en el proyecto.

### `frontend/src/app/__init__.py`

Marcador de paquete / m?dulo hist?rico.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.


### `frontend/src/app/async_runner.py`

Puente asyncio/Tkinter: red en un hilo; todos los widgets en el hilo Tk.

SHA-256: `380e943b496de1a35b27cd20cbe685093d9f96c59df61be5f4792e5153f8c254`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `AsyncRunner` | 9?119 | Entrega resultados a Tk mediante una cola revisada con after. |
| `AsyncRunner.__init__` | 12?23 | root: ventana principal que programa la revisión de resultados. |
| `AsyncRunner._run` | 25?33 | Mantiene un único loop para todas las conexiones HTTP. |
| `AsyncRunner.submit` | 35?44 | coroutine: tarea; owner: widget dueño; callbacks: resultado/error en Tk. |
| `AsyncRunner.cancel_owner` | 46?51 | owner: vista/modal cerrado; elimina sus callbacks y cancela peticiones. |
| `AsyncRunner._poll` | 53?76 | Procesa resultados en el hilo principal; ignora ventanas destruidas. |
| `AsyncRunner.close` | 78?112 | cleanup: coroutine que cierra HTTP; on_closed: cierre de ventana en Tk. |
| `AsyncRunner.close.shutdown` | 89?96 | Espera cancelaciones y libera HTTP antes de detener el loop. |
| `AsyncRunner.close.finish` | 100?110 | Revisa el cierre sin bloquear el bucle de eventos de Tk. |
| `AsyncRunner._wait_thread` | 114?119 | on_closed: se llama cuando el trabajador ya terminó. |

**C?digo de referencia, no ejecutable**

```python
"""Puente asyncio/Tkinter: red en un hilo; todos los widgets en el hilo Tk."""
import asyncio
from concurrent.futures import CancelledError
from queue import Empty, Queue
from threading import Event, Thread
import logging


class AsyncRunner:
    """Entrega resultados a Tk mediante una cola revisada con after."""

    def __init__(self, root):
        """root: ventana principal que programa la revisión de resultados."""
        self.root = root
        self.loop = asyncio.new_event_loop()
        self._ready = Event()
        self._queue = Queue()
        self._pending = {}
        self._closing = False
        self._thread = Thread(target=self._run, name='fixify-asyncio', daemon=True)
        self._thread.start()
        self._ready.wait()
        self._poll_id = root.after(30, self._poll)

    def _run(self):
        """Mantiene un único loop para todas las conexiones HTTP."""
        asyncio.set_event_loop(self.loop)
        self._ready.set()
        try:
            self.loop.run_forever()
        finally:
            self.loop.run_until_complete(self.loop.shutdown_asyncgens())
            self.loop.close()

    def submit(self, coroutine, owner, on_success, on_error):
        """coroutine: tarea; owner: widget dueño; callbacks: resultado/error en Tk."""
        if self._closing:
            coroutine.close()
            return None
        future = asyncio.run_coroutine_threadsafe(coroutine, self.loop)
        self._pending[future] = (owner, on_success, on_error)
        # Este callback solo toca una cola Python; nunca llama a Tkinter.
        future.add_done_callback(self._queue.put)
        return future

    def cancel_owner(self, owner):
        """owner: vista/modal cerrado; elimina sus callbacks y cancela peticiones."""
        for future, context in list(self._pending.items()):
            if context[0] is owner:
                self._pending.pop(future, None)
                future.cancel()

    def _poll(self):
        """Procesa resultados en el hilo principal; ignora ventanas destruidas."""
        try:
            while True:
                future = self._queue.get_nowait()
                context = self._pending.pop(future, None)
                if context is None:
                    continue
                owner, success, error = context
                if not owner.winfo_exists():
                    continue
                try:
                    result = future.result()
                except CancelledError:
                    continue
                except Exception as exc:
                    error(exc)
                else:
                    success(result)
        except Empty:
            pass
        finally:
            if not self._closing:
                self._poll_id = self.root.after(30, self._poll)

    def close(self, cleanup, on_closed):
        """cleanup: coroutine que cierra HTTP; on_closed: cierre de ventana en Tk."""
        if self._closing:
            cleanup.close()
            return
        self._closing = True
        self.root.after_cancel(self._poll_id)
        for future in self._pending:
            future.cancel()
        self._pending.clear()

        async def shutdown():
            """Espera cancelaciones y libera HTTP antes de detener el loop."""
            current = asyncio.current_task()
            tasks = [task for task in asyncio.all_tasks() if task is not current]
            for task in tasks:
                task.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)
            await cleanup

        future = asyncio.run_coroutine_threadsafe(shutdown(), self.loop)

        def finish():
            """Revisa el cierre sin bloquear el bucle de eventos de Tk."""
            if not future.done():
                self.root.after(30, finish)
                return
            try:
                future.result()
            except Exception:
                logging.exception('No se pudo cerrar el cliente HTTP')
            self.loop.call_soon_threadsafe(self.loop.stop)
            self._wait_thread(on_closed)

        self.root.after(30, finish)

    def _wait_thread(self, on_closed):
        """on_closed: se llama cuando el trabajador ya terminó."""
        if self._thread.is_alive():
            self.root.after(30, lambda: self._wait_thread(on_closed))
        else:
            on_closed()
```

### `frontend/src/app/main.py`

Entrada de escritorio: python frontend/src/app/main.py desde la raíz.

SHA-256: `edc8a4bd6c81f70f4aacf64c2e15ec0b994d7a3ac3b1a448b0bf5ba9ee109f16`.


**C?digo de referencia, no ejecutable**

```python
"""Entrada de escritorio: python frontend/src/app/main.py desde la raíz."""
from pathlib import Path
import sys

# Permite imports de app, services y ui al ejecutar este archivo directamente.
RUTA_SRC = str(Path(__file__).resolve().parents[1])
if RUTA_SRC not in sys.path:
    sys.path.insert(0, RUTA_SRC)

from ui.login import VentanaLogin


if __name__ == '__main__':
    app = VentanaLogin()
    app.mainloop()
```

### `frontend/src/models/__init__.py`

Contratos Pydantic de entrada/salida, separados por módulo de negocio.

SHA-256: `0f07cbabe644a614588ea50219eb23d87f4dd2c7b75ad4d4036861bdba357eeb`.


**C?digo de referencia, no ejecutable**

```python
"""Contratos Pydantic de entrada/salida, separados por módulo de negocio."""
```

### `frontend/src/models/auth.py`

Credenciales de entrada y sesión devuelta por /auth/login.

SHA-256: `5b0bb869e5fc39092b280ab9919e289d4eb3e732fbd8eab40fe516e1c5094e51`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `Login` | 7?10 | username/password: credenciales; password se oculta en la representación. |
| `Sesion` | 13?18 | access_token: JWT; token_type: esquema Bearer; role: permisos; message: confirmación. |

**C?digo de referencia, no ejecutable**

```python
"""Credenciales de entrada y sesión devuelta por /auth/login."""
from typing import Literal
from pydantic import Field
from models.base import Contrato, Rol


class Login(Contrato):
    """username/password: credenciales; password se oculta en la representación."""
    username: str = Field(min_length=1)
    password: str = Field(min_length=1, repr=False)


class Sesion(Contrato):
    """access_token: JWT; token_type: esquema Bearer; role: permisos; message: confirmación."""
    access_token: str = Field(min_length=1, repr=False)
    token_type: Literal['bearer']
    role: Rol
    message: str | None = None
```

### `frontend/src/models/base.py`

Configuración común del contrato; no contiene acceso HTTP ni a la BD.

SHA-256: `062551390290430148348056b390c9d7e6db48f804048b267c7b2e4d61cb8483`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `Contrato` | 8?10 | Valida tipos sin conversiones silenciosas e ignora campos extra del servidor. |
| `Mensaje` | 13?15 | Confirmación de operaciones: mensaje es el texto enviado por el backend. |

**C?digo de referencia, no ejecutable**

```python
"""Configuración común del contrato; no contiene acceso HTTP ni a la BD."""
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field

Rol = Literal['admin', 'tecnico', 'recepcion']


class Contrato(BaseModel):
    """Valida tipos sin conversiones silenciosas e ignora campos extra del servidor."""
    model_config = ConfigDict(strict=True, extra='ignore', frozen=True)


class Mensaje(Contrato):
    """Confirmación de operaciones: mensaje es el texto enviado por el backend."""
    mensaje: str = Field(min_length=1)
```

### `frontend/src/models/clients.py`

Contrato de clientes: datos para alta, registro consultado y confirmación.

SHA-256: `45da089d667c597b4dbe6e973c0ef4ccd7e6c2325cdd6aa42624a9749f5ffb82`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `ClienteCrear` | 6?11 | nombre/telefono: obligatorios; correo/direccion: contacto opcional. |
| `Cliente` | 14?20 | Registro leído; id es el identificador API, no el nombre físico de columna. |
| `ClienteRegistrado` | 23?25 | Confirmación de alta; cliente contiene el registro persistido. |

**C?digo de referencia, no ejecutable**

```python
"""Contrato de clientes: datos para alta, registro consultado y confirmación."""
from pydantic import Field
from models.base import Contrato, Mensaje


class ClienteCrear(Contrato):
    """nombre/telefono: obligatorios; correo/direccion: contacto opcional."""
    nombre: str = Field(min_length=2, max_length=100)
    telefono: str = Field(min_length=8, max_length=20)
    correo: str | None = None
    direccion: str | None = Field(default=None, max_length=250)


class Cliente(Contrato):
    """Registro leído; id es el identificador API, no el nombre físico de columna."""
    id: int = Field(gt=0)
    nombre: str
    telefono: str
    correo: str | None = None
    direccion: str | None = None


class ClienteRegistrado(Mensaje):
    """Confirmación de alta; cliente contiene el registro persistido."""
    cliente: Cliente
```

### `frontend/src/models/database.py`

Marcador de paquete / m?dulo hist?rico.

SHA-256: `9eb4cd724e929b30519c145a718ca9dfaa0d828a11efd2819d42bf8ad52b8b56`.


### `frontend/src/models/equipment.py`

Contrato de equipos; mantiene los nombres públicos de la API.

SHA-256: `02a5e221d23a0ee093903144d060f8e03bc4f9af4d473a7b7d27babab987aca7`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `EquipoCrear` | 6?13 | Características, falla y client_id del propietario; serie opcional. |
| `Equipo` | 16?24 | Registro consultado: id, características, falla y propietario. |
| `DetallesEquipo` | 27?33 | Resumen del POST; usa id_interno/cliente_id según la respuesta actual. |
| `EquipoRegistrado` | 36?39 | Confirmación de alta: folio visual y resumen del equipo guardado. |

**C?digo de referencia, no ejecutable**

```python
"""Contrato de equipos; mantiene los nombres públicos de la API."""
from pydantic import Field
from models.base import Contrato, Mensaje


class EquipoCrear(Contrato):
    """Características, falla y client_id del propietario; serie opcional."""
    tipo: str = Field(min_length=2, max_length=50)
    marca: str = Field(min_length=2, max_length=50)
    modelo: str = Field(min_length=2, max_length=50)
    problema_reportado: str = Field(min_length=5)
    client_id: int = Field(gt=0)
    numero_serie: str | None = Field(default=None, max_length=100)


class Equipo(Contrato):
    """Registro consultado: id, características, falla y propietario."""
    id: int = Field(gt=0)
    tipo: str
    marca: str
    modelo: str
    problema_reportado: str
    client_id: int = Field(gt=0)
    numero_serie: str | None = None


class DetallesEquipo(Contrato):
    """Resumen del POST; usa id_interno/cliente_id según la respuesta actual."""
    id_interno: int = Field(gt=0)
    tipo: str
    marca: str
    modelo: str
    cliente_id: int = Field(gt=0)


class EquipoRegistrado(Mensaje):
    """Confirmación de alta: folio visual y resumen del equipo guardado."""
    folio: str = Field(min_length=1)
    detalles: DetallesEquipo
```

### `frontend/src/models/users.py`

Contrato de personal, alta y cambios de rol; nunca conserva hashes.

SHA-256: `22592fb3e3a0e0eaf70c1f17f70e0b698afb6fafbec7e289c11f84b7637ded2e`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `UsuarioCrear` | 6?10 | username/password: credenciales; role: rol autorizado de dominio. |
| `Usuario` | 13?18 | Registro público: ID, nombre de cuenta, rol y estado de acceso. |
| `RolActualizar` | 21?23 | role: nuevo rol de la cuenta seleccionada. |
| `UsuarioRegistrado` | 26?28 | usuario: username confirmado por POST /usuarios/. |

**C?digo de referencia, no ejecutable**

```python
"""Contrato de personal, alta y cambios de rol; nunca conserva hashes."""
from pydantic import Field
from models.base import Contrato, Mensaje, Rol


class UsuarioCrear(Contrato):
    """username/password: credenciales; role: rol autorizado de dominio."""
    username: str = Field(min_length=4, max_length=50)
    password: str = Field(min_length=6, repr=False)
    role: Rol = 'tecnico'


class Usuario(Contrato):
    """Registro público: ID, nombre de cuenta, rol y estado de acceso."""
    id: int = Field(gt=0)
    username: str
    role: Rol
    is_active: bool


class RolActualizar(Contrato):
    """role: nuevo rol de la cuenta seleccionada."""
    role: Rol


class UsuarioRegistrado(Mensaje):
    """usuario: username confirmado por POST /usuarios/."""
    usuario: str
```

### `frontend/src/services/__init__.py`

Construye los servicios con una sola sesión HTTP compartida.

SHA-256: `e0d180505ab2ada528d68051fa3b7c4e6a1411d1344960ba6867b65ff76b2621`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `Services` | 9?18 | Contenedor inyectable para evitar clientes globales en las vistas. |
| `Services.__init__` | 12?18 | api: transporte opcional; si se omite utiliza la configuración local. |

**C?digo de referencia, no ejecutable**

```python
"""Construye los servicios con una sola sesión HTTP compartida."""
from services.api_client import ApiClient
from services.auth_service import AuthService
from services.clients_service import ClientsService
from services.equipment_service import EquipmentService
from services.users_service import UsersService


class Services:
    """Contenedor inyectable para evitar clientes globales en las vistas."""

    def __init__(self, api=None):
        """api: transporte opcional; si se omite utiliza la configuración local."""
        self.api = api if api is not None else ApiClient()
        self.auth = AuthService(self.api)
        self.clients = ClientsService(self.api)
        self.equipment = EquipmentService(self.api)
        self.users = UsersService(self.api)
```

### `frontend/src/services/api_client.py`

Transporte HTTP compartido; las rutas de negocio viven en cada service.

SHA-256: `1b30a0211d10128126a57b4f04ee0b49d75380df29b3ce44b4f2cf13c829dac3`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `ApiError` | 11?16 | Error legible de API; status_code identifica errores HTTP, si existen. |
| `ApiError.__init__` | 14?16 | Ver m?dulo. |
| `ApiClient` | 19?74 | Reutiliza conexiones HTTP y el token de la sesión de escritorio. |
| `ApiClient.__init__` | 22?27 | base_url: servidor; transport: transporte opcional para pruebas sin red. |
| `ApiClient.set_token` | 29?31 | token: JWT vigente o None para cerrar la sesión local. |
| `ApiClient.request` | 33?55 | method/path: operación HTTP (PUT, POST, GET, DELETE); kwargs: JSON, formulario o parámetros. |
| `ApiClient._error_message` | 58?68 | response: respuesta HTTP; convierte detail de FastAPI a texto breve. |
| `ApiClient.aclose` | 70?74 | Libera conexiones al cerrar la aplicación. |
| `ModuleService` | 77?99 | Base para compartir transporte y validar listas sin duplicar HTTP. |
| `ModuleService.__init__` | 80?82 | api: ApiClient compartido por los módulos. |
| `ModuleService._parse` | 85?92 | model: contrato esperado; data: valores; entrada: distingue formulario de respuesta. |
| `ModuleService._list` | 94?99 | path: colección; model: contrato de cada registro; kwargs: filtros opcionales. |

**C?digo de referencia, no ejecutable**

```python
"""Transporte HTTP compartido; las rutas de negocio viven en cada service."""
import os
import httpx
from typing import TypeVar
from pydantic import ValidationError
from models.base import Contrato

Modelo = TypeVar('Modelo', bound=Contrato)


class ApiError(Exception):
    """Error legible de API; status_code identifica errores HTTP, si existen."""

    def __init__(self, message, status_code=None):
        super().__init__(message)
        self.status_code = status_code


class ApiClient:
    """Reutiliza conexiones HTTP y el token de la sesión de escritorio."""

    def __init__(self, base_url=None, transport=None):
        """base_url: servidor; transport: transporte opcional para pruebas sin red."""
        self.base_url = base_url or os.getenv('FIXIFY_API_URL', 'http://127.0.0.1:8000')
        self.token = None
        self._transport = transport
        self._client = None

    def set_token(self, token):
        """token: JWT vigente o None para cerrar la sesión local."""
        self.token = token

    async def request(self, method, path, *, authenticated=True, **kwargs):
        """method/path: operación HTTP (PUT, POST, GET, DELETE); kwargs: JSON, formulario o parámetros."""
        # Se crea y utiliza exclusivamente en el loop asyncio del trabajador.
        if self._client is None:
            self._client = httpx.AsyncClient(
                base_url=self.base_url, timeout=5, transport=self._transport,
                follow_redirects=True,
            )
        headers = {'Authorization': f'Bearer {self.token}'} if authenticated and self.token else {}
        try:
            response = await self._client.request(method, path, headers=headers, **kwargs)
        except httpx.TimeoutException as exc:
            raise ApiError('El servidor tardó demasiado en responder.') from exc
        except httpx.RequestError as exc:
            raise ApiError('No se pudo conectar con el servidor.') from exc
        if response.is_error:
            raise ApiError(self._error_message(response), response.status_code)
        if response.status_code == 204:
            return None
        try:
            return response.json()
        except ValueError as exc:
            raise ApiError('El servidor devolvió una respuesta inválida.') from exc

    @staticmethod
    def _error_message(response):
        """response: respuesta HTTP; convierte detail de FastAPI a texto breve."""
        try:
            detail = response.json().get('detail')
            if isinstance(detail, list):
                return '\n'.join(str(error.get('msg', 'Dato inválido')) for error in detail)
            if isinstance(detail, str):
                return detail
        except (ValueError, AttributeError, TypeError):
            pass
        return f'La solicitud falló (HTTP {response.status_code}).'

    async def aclose(self):
        """Libera conexiones al cerrar la aplicación."""
        if self._client is not None:
            await self._client.aclose()
            self._client = None


class ModuleService:
    """Base para compartir transporte y validar listas sin duplicar HTTP."""

    def __init__(self, api):
        """api: ApiClient compartido por los módulos."""
        self.api = api

    @staticmethod
    def _parse(model: type[Modelo], data, *, entrada=False) -> Modelo:
        """model: contrato esperado; data: valores; entrada: distingue formulario de respuesta."""
        try:
            return model.model_validate(data)
        except ValidationError:
            # No incluimos valores inválidos: podrían contener contraseñas o tokens.
            origen = 'Los datos ingresados' if entrada else 'La respuesta del servidor'
            raise ApiError(f'{origen} no cumplen el contrato {model.__name__}.') from None

    async def _list(self, path, model: type[Modelo], **kwargs) -> list[Modelo]:
        """path: colección; model: contrato de cada registro; kwargs: filtros opcionales."""
        data = await self.api.request('GET', path, **kwargs)
        if not isinstance(data, list) or any(not isinstance(item, dict) for item in data):
            raise ApiError('El servidor no devolvió una lista válida.')
        return [self._parse(model, item) for item in data]
```

### `frontend/src/services/auth_service.py`

Operaciones de autenticación; no modifica widgets ni guarda contraseñas.

SHA-256: `acc0d9b36e0a5e271ea023cb947f3c06745d4ac93e923c21f5d9b5fa0013aea6`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `AuthService` | 6?14 | Solicita credenciales de sesión al backend. |
| `AuthService.login` | 9?14 | username/password: credenciales; retorna token y rol validados. |

**C?digo de referencia, no ejecutable**

```python
"""Operaciones de autenticación; no modifica widgets ni guarda contraseñas."""
from services.api_client import ModuleService
from models.auth import Login, Sesion


class AuthService(ModuleService):
    """Solicita credenciales de sesión al backend."""

    async def login(self, username, password) -> Sesion:
        """username/password: credenciales; retorna token y rol validados."""
        entrada = self._parse(Login, {'username': username, 'password': password}, entrada=True)
        data = await self.api.request('POST', '/auth/login', authenticated=False, data=entrada.model_dump())
        # La UI guarda el token solo si aún existe la ventana que pidió el login.
        return self._parse(Sesion, data)
```

### `frontend/src/services/clients_service.py`

Rutas de clientes y transformación de sus datos de entrada.

SHA-256: `ca94ecb8a906bb095d6a8747fea755ebdf9e0e897d32751d3a9de9fa3d99f330`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `ClientsService` | 6?24 | Consulta y registra clientes mediante la API. |
| `ClientsService.listar` | 9?11 | Retorna los clientes permitidos para la sesión actual. |
| `ClientsService.crear` | 13?20 | nombre/telefono: obligatorios; correo/direccion: datos opcionales. |
| `ClientsService.buscar` | 22?24 | termino: parte del nombre o teléfono que se desea localizar. |

**C?digo de referencia, no ejecutable**

```python
"""Rutas de clientes y transformación de sus datos de entrada."""
from services.api_client import ModuleService
from models.clients import Cliente, ClienteCrear, ClienteRegistrado


class ClientsService(ModuleService):
    """Consulta y registra clientes mediante la API."""

    async def listar(self) -> list[Cliente]:
        """Retorna los clientes permitidos para la sesión actual."""
        return await self._list('/clientes/', Cliente)

    async def crear(self, nombre, telefono, correo=None, direccion=None) -> ClienteRegistrado:
        """nombre/telefono: obligatorios; correo/direccion: datos opcionales."""
        entrada = self._parse(ClienteCrear, {
            'nombre': nombre, 'telefono': telefono,
            'correo': correo or None, 'direccion': direccion or None,
        }, entrada=True)
        data = await self.api.request('POST', '/clientes/', json=entrada.model_dump())
        return self._parse(ClienteRegistrado, data)

    async def buscar(self, termino) -> list[Cliente]:
        """termino: parte del nombre o teléfono que se desea localizar."""
        return await self._list('/clientes/buscar', Cliente, params={'q': termino})
```

### `frontend/src/services/equipment_service.py`

Operaciones del módulo de equipos; mantiene el contrato actual de API.

SHA-256: `c6ddbb6ab00667d912af1b750eaf5c952de6678d79ba6166d50542a45a1f06b8`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `EquipmentService` | 6?21 | Consulta y registra equipos asociados a un cliente. |
| `EquipmentService.listar` | 9?11 | Retorna equipos mediante GET /equipos/. |
| `EquipmentService.crear` | 13?21 | tipo/marca/modelo: equipo; problema_reportado: falla; client_id: dueño; numero_serie: opcional. |

**C?digo de referencia, no ejecutable**

```python
"""Operaciones del módulo de equipos; mantiene el contrato actual de API."""
from services.api_client import ModuleService
from models.equipment import Equipo, EquipoCrear, EquipoRegistrado


class EquipmentService(ModuleService):
    """Consulta y registra equipos asociados a un cliente."""

    async def listar(self) -> list[Equipo]:
        """Retorna equipos mediante GET /equipos/."""
        return await self._list('/equipos/', Equipo)

    async def crear(self, tipo, marca, modelo, problema_reportado, client_id, numero_serie=None) -> EquipoRegistrado:
        """tipo/marca/modelo: equipo; problema_reportado: falla; client_id: dueño; numero_serie: opcional."""
        entrada = self._parse(EquipoCrear, {
            'tipo': tipo, 'marca': marca, 'modelo': modelo,
            'problema_reportado': problema_reportado, 'client_id': client_id,
            'numero_serie': numero_serie or None,
        }, entrada=True)
        data = await self.api.request('POST', '/equipos/', json=entrada.model_dump())
        return self._parse(EquipoRegistrado, data)
```

### `frontend/src/services/users_service.py`

Rutas de administración de usuarios, independientes de las otras entidades.

SHA-256: `32968e44c61a09a75be3d82ac334215033f9c611a47bf3c4f1e234267fe3ca35`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `UsersService` | 7?32 | Operaciones de usuarios autorizadas por el backend para admin. |
| `UsersService.listar` | 10?12 | Retorna el personal registrado. |
| `UsersService.crear` | 14?20 | username/password: credenciales; role: admin, tecnico o recepcion. |
| `UsersService.cambiar_rol` | 22?26 | user_id: cuenta destino; role: nuevo rol de dominio. |
| `UsersService.activar` | 28?32 | user_id: cuenta destino; activo: habilitar o inhabilitar acceso. |

**C?digo de referencia, no ejecutable**

```python
"""Rutas de administración de usuarios, independientes de las otras entidades."""
from services.api_client import ModuleService
from models.base import Mensaje
from models.users import Usuario, UsuarioCrear, UsuarioRegistrado, RolActualizar


class UsersService(ModuleService):
    """Operaciones de usuarios autorizadas por el backend para admin."""

    async def listar(self) -> list[Usuario]:
        """Retorna el personal registrado."""
        return await self._list('/usuarios/', Usuario)

    async def crear(self, username, password, role) -> UsuarioRegistrado:
        """username/password: credenciales; role: admin, tecnico o recepcion."""
        entrada = self._parse(UsuarioCrear, {
            'username': username, 'password': password, 'role': role,
        }, entrada=True)
        data = await self.api.request('POST', '/usuarios/', json=entrada.model_dump())
        return self._parse(UsuarioRegistrado, data)

    async def cambiar_rol(self, user_id, role) -> Mensaje:
        """user_id: cuenta destino; role: nuevo rol de dominio."""
        entrada = self._parse(RolActualizar, {'role': role}, entrada=True)
        data = await self.api.request('PATCH', f'/usuarios/{user_id}/rol', json=entrada.model_dump())
        return self._parse(Mensaje, data)

    async def activar(self, user_id, activo=True) -> Mensaje:
        """user_id: cuenta destino; activo: habilitar o inhabilitar acceso."""
        action = 'habilitar' if activo else 'inhabilitar'
        data = await self.api.request('PATCH', f'/usuarios/{user_id}/{action}')
        return self._parse(Mensaje, data)
```

### `frontend/src/ui/__init__.py`

Marcador de paquete / m?dulo hist?rico.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.


### `frontend/src/ui/components.py`

Controles reutilizables para tablas, formularios y solicitudes asíncronas.

SHA-256: `bb8dfdb4bf123ca11e07ec69dfbc37b165c0923783d198b3afbad3b934e27a90`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `Field` | 10?16 | name: clave API; label: texto; required: obligatorio; secret: contraseña; choices: opciones. |
| `AsyncFrame` | 19?30 | Base de vista que cancela sus peticiones cuando se destruye. |
| `AsyncFrame.__init__` | 22?25 | master: contenedor; services: módulos API; runner: puente asyncio; role: rol de dominio. |
| `AsyncFrame.destroy` | 27?30 | Descarta tareas y callbacks antes de destruir los controles. |
| `create_table` | 33?45 | parent: contenedor; columns: encabezados; height: filas visibles; retorna Treeview. |
| `FormDialog` | 48?104 | Formulario genérico con validación de campos requeridos y guardado async. |
| `FormDialog.__init__` | 51?77 | master: vista; title: título; fields: campos; runner: tareas; save/on_saved: guardar/confirmar. |
| `FormDialog._submit` | 79?89 | Lee controles en Tk; conserva espacios de contraseñas y evita doble envío. |
| `FormDialog._saved` | 91?94 | result: respuesta API; notifica al módulo y cierra el formulario. |
| `FormDialog._failed` | 96?99 | error: fallo de solicitud; permite corregir y reintentar. |
| `FormDialog.destroy` | 101?104 | Cancela el guardado pendiente y evita callbacks sobre un modal cerrado. |
| `CollectionView` | 107?179 | Comparte listado, recarga, alta y estados visuales entre módulos. |
| `CollectionView.__init__` | 113?129 | Recibe dependencias comunes; cada subclase define columnas y operaciones. |
| `CollectionView.fetch` | 131?133 | Retorna registros; la subclase elige su service. |
| `CollectionView.save` | 135?137 | values: campos del formulario; la subclase llama al service de alta. |
| `CollectionView.row` | 139?141 | record: entidad API; la subclase la convierte a valores de tabla. |
| `CollectionView.load` | 143?149 | Consulta sin bloquear; el botón impide recargas duplicadas. |
| `CollectionView._loaded` | 151?159 | records: lista validada; reemplaza filas y actualiza estado. |
| `CollectionView._failed` | 161?164 | error: fallo visible; conserva datos anteriores y habilita reintento. |
| `CollectionView.open_form` | 166?171 | Abre el formulario común con los campos del módulo. |
| `CollectionView._created` | 173?179 | result: confirmación de alta; informa y vuelve a consultar registros. |

**C?digo de referencia, no ejecutable**

```python
"""Controles reutilizables para tablas, formularios y solicitudes asíncronas."""
from dataclasses import dataclass
from tkinter import messagebox
import ttkbootstrap as ttk
from models.base import Mensaje
from models.equipment import EquipoRegistrado


@dataclass(frozen=True)
class Field:
    """name: clave API; label: texto; required: obligatorio; secret: contraseña; choices: opciones."""
    name: str
    label: str
    required: bool = True
    secret: bool = False
    choices: tuple = ()


class AsyncFrame(ttk.Frame):
    """Base de vista que cancela sus peticiones cuando se destruye."""

    def __init__(self, master, services, runner, role):
        """master: contenedor; services: módulos API; runner: puente asyncio; role: rol de dominio."""
        super().__init__(master)
        self.services, self.runner, self.role = services, runner, role

    def destroy(self):
        """Descarta tareas y callbacks antes de destruir los controles."""
        self.runner.cancel_owner(self)
        super().destroy()


def create_table(parent, columns, height=12):
    """parent: contenedor; columns: encabezados; height: filas visibles; retorna Treeview."""
    frame = ttk.Frame(parent)
    frame.pack(fill='both', expand=True)
    table = ttk.Treeview(frame, columns=columns, show='headings', height=height)
    for column in columns:
        table.heading(column, text=column)
        table.column(column, anchor='center', width=130)
    scroll = ttk.Scrollbar(frame, orient='vertical', command=table.yview)
    table.configure(yscrollcommand=scroll.set)
    table.pack(side='left', fill='both', expand=True)
    scroll.pack(side='right', fill='y')
    return table


class FormDialog(ttk.Toplevel):
    """Formulario genérico con validación de campos requeridos y guardado async."""

    def __init__(self, master, title, fields, runner, save, on_saved):
        """master: vista; title: título; fields: campos; runner: tareas; save/on_saved: guardar/confirmar."""
        super().__init__(master)
        self.runner, self.save, self.on_saved = runner, save, on_saved
        self.fields, self.inputs = fields, {}
        self.title(title)
        self.resizable(False, False)
        body = ttk.Frame(self, padding=20)
        body.pack(fill='both', expand=True)
        ttk.Label(body, text=title, font=('Arial', 14, 'bold')).pack(pady=(0, 15))
        for field in fields:
            ttk.Label(body, text=field.label).pack(anchor='w', pady=(5, 0))
            if field.choices:
                widget = ttk.Combobox(body, values=field.choices, state='readonly', width=38)
                widget.current(0)
            else:
                widget = ttk.Entry(body, width=40, show='*' if field.secret else '')
            widget.pack(fill='x', pady=5)
            self.inputs[field.name] = widget
        self.status = ttk.Label(body, text='', wraplength=350)
        self.status.pack(fill='x', pady=5)
        self.button = ttk.Button(body, text='Guardar', bootstyle='success', command=self._submit)
        self.button.pack(pady=(10, 0))
        self.transient(master.winfo_toplevel())
        self.protocol('WM_DELETE_WINDOW', self.destroy)
        self.update_idletasks()
        self.place_window_center()

    def _submit(self):
        """Lee controles en Tk; conserva espacios de contraseñas y evita doble envío."""
        values = {f.name: self.inputs[f.name].get() if f.secret
                  else self.inputs[f.name].get().strip() for f in self.fields}
        missing = [f.label for f in self.fields if f.required and not values[f.name]]
        if missing:
            self.status.configure(text='Complete: ' + ', '.join(missing))
            return
        self.button.configure(state='disabled')
        self.status.configure(text='Guardando…')
        self.runner.submit(self.save(values), self, self._saved, self._failed)

    def _saved(self, result):
        """result: respuesta API; notifica al módulo y cierra el formulario."""
        self.on_saved(result)
        self.destroy()

    def _failed(self, error):
        """error: fallo de solicitud; permite corregir y reintentar."""
        self.status.configure(text=str(error))
        self.button.configure(state='normal')

    def destroy(self):
        """Cancela el guardado pendiente y evita callbacks sobre un modal cerrado."""
        self.runner.cancel_owner(self)
        super().destroy()


class CollectionView(AsyncFrame):
    """Comparte listado, recarga, alta y estados visuales entre módulos."""
    title = ''
    columns = ()
    fields = ()

    def __init__(self, master, services, runner, role):
        """Recibe dependencias comunes; cada subclase define columnas y operaciones."""
        super().__init__(master, services, runner, role)
        self.records = []
        self._load_future = None
        self._dialog = None
        ttk.Label(self, text=self.title, font=('Arial', 18, 'bold')).pack(anchor='w', pady=(0, 15))
        bar = ttk.Frame(self)
        bar.pack(fill='x', pady=(0, 10))
        self.refresh = ttk.Button(bar, text='Actualizar', command=self.load)
        self.refresh.pack(side='left')
        self.create = ttk.Button(bar, text='+ Nuevo', bootstyle='success', command=self.open_form)
        self.create.pack(side='right')
        self.status = ttk.Label(self, text='', wraplength=750)
        self.status.pack(fill='x', pady=(0, 10))
        self.table = create_table(self, self.columns)
        self.load()

    async def fetch(self):
        """Retorna registros; la subclase elige su service."""
        raise NotImplementedError

    async def save(self, values):
        """values: campos del formulario; la subclase llama al service de alta."""
        raise NotImplementedError

    def row(self, record):
        """record: entidad API; la subclase la convierte a valores de tabla."""
        raise NotImplementedError

    def load(self):
        """Consulta sin bloquear; el botón impide recargas duplicadas."""
        if self._load_future is not None:
            self._load_future.cancel()
        self.refresh.configure(state='disabled')
        self.status.configure(text='Cargando…')
        self._load_future = self.runner.submit(self.fetch(), self, self._loaded, self._failed)

    def _loaded(self, records):
        """records: lista validada; reemplaza filas y actualiza estado."""
        self.records = records
        for item in self.table.get_children():
            self.table.delete(item)
        for index, record in enumerate(records):
            self.table.insert('', 'end', iid=str(index), values=self.row(record))
        self.status.configure(text=f'{len(records)} registros')
        self.refresh.configure(state='normal')

    def _failed(self, error):
        """error: fallo visible; conserva datos anteriores y habilita reintento."""
        self.status.configure(text=str(error))
        self.refresh.configure(state='normal')

    def open_form(self):
        """Abre el formulario común con los campos del módulo."""
        if self._dialog is not None and self._dialog.winfo_exists():
            self._dialog.lift()
            return
        self._dialog = FormDialog(self, self.title, self.fields, self.runner, self.save, self._created)

    def _created(self, result: Mensaje):
        """result: confirmación de alta; informa y vuelve a consultar registros."""
        folio = result.folio if isinstance(result, EquipoRegistrado) else None
        messagebox.showinfo('Registro guardado', f'Guardado correctamente. Folio: {folio}'
                            if folio else 'Guardado correctamente.', parent=self)
        # load cancela solo la lectura previa; no interrumpe otras escrituras.
        self.load()
```

### `frontend/src/ui/login.py`

Login de escritorio con solicitudes asíncronas y sesión en memoria.

SHA-256: `75b1c6489812e3b7db389104be7f4f592cb7d5d33ac1db04e6a2c3ab8f9f8c36`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `VentanaLogin` | 7?82 | Ventana principal; administra recursos y el ciclo de vida de la aplicación. |
| `VentanaLogin.__init__` | 10?19 | services: contenedor opcional para inyectar una API de pruebas. |
| `VentanaLogin.crear_widgets` | 21?38 | Construye campos con etiquetas estables y feedback de conexión. |
| `VentanaLogin.verificar_login` | 40?52 | Valida presencia; preserva contraseña y evita solicitudes repetidas. |
| `VentanaLogin._logged_in` | 54?62 | username: cuenta ingresada; data: respuesta validada con token y rol. |
| `VentanaLogin._failed` | 64?67 | error: fallo de login; deja corregir credenciales o reintentar. |
| `VentanaLogin.mostrar_recuperacion` | 69?72 | Informa el procedimiento actual; no existe recuperación automática. |
| `VentanaLogin.cerrar_aplicacion` | 74?82 | Cancela tareas y cierra HTTP antes de destruir la ventana principal. |

**C?digo de referencia, no ejecutable**

```python
"""Login de escritorio con solicitudes asíncronas y sesión en memoria."""
import ttkbootstrap as ttk
from app.async_runner import AsyncRunner
from services import Services


class VentanaLogin(ttk.Window):
    """Ventana principal; administra recursos y el ciclo de vida de la aplicación."""

    def __init__(self, services=None):
        """services: contenedor opcional para inyectar una API de pruebas."""
        super().__init__(title='Fixify — Iniciar sesión', themename='litera', size=(450, 460))
        self.services = services if services is not None else Services()
        self.runner = AsyncRunner(self)
        self._closing = False
        self.resizable(False, False)
        self.protocol('WM_DELETE_WINDOW', self.cerrar_aplicacion)
        self.crear_widgets()
        self.place_window_center()

    def crear_widgets(self):
        """Construye campos con etiquetas estables y feedback de conexión."""
        body = ttk.Frame(self, padding=35)
        body.pack(fill='both', expand=True)
        ttk.Label(body, text='FIXIFY', font=('Arial', 18, 'bold')).pack(pady=(0, 20))
        ttk.Label(body, text='Usuario').pack(anchor='w')
        self.entry_usuario = ttk.Entry(body)
        self.entry_usuario.pack(fill='x', pady=(5, 15))
        ttk.Label(body, text='Contraseña').pack(anchor='w')
        self.entry_password = ttk.Entry(body, show='*')
        self.entry_password.pack(fill='x', pady=5)
        self.status = ttk.Label(body, text='', wraplength=370)
        self.status.pack(fill='x', pady=10)
        self.button = ttk.Button(body, text='Iniciar sesión', command=self.verificar_login)
        self.button.pack(fill='x', pady=10)
        ttk.Button(body, text='¿Olvidaste tu contraseña?', bootstyle='link',
                   command=self.mostrar_recuperacion).pack()
        self.bind('<Return>', lambda event: self.verificar_login())

    def verificar_login(self):
        """Valida presencia; preserva contraseña y evita solicitudes repetidas."""
        if self._closing or str(self.button['state']) == 'disabled':
            return
        username = self.entry_usuario.get().strip()
        password = self.entry_password.get()
        if not username or not password:
            self.status.configure(text='Complete usuario y contraseña.')
            return
        self.button.configure(state='disabled')
        self.status.configure(text='Conectando…')
        self.runner.submit(self.services.auth.login(username, password), self,
                           lambda data: self._logged_in(username, data), self._failed)

    def _logged_in(self, username, data):
        """username: cuenta ingresada; data: respuesta validada con token y rol."""
        from ui.sistema_gestion import SistemaGestion
        self.services.api.set_token(data.access_token)
        self.entry_password.delete(0, 'end')
        self.status.configure(text='')
        self.button.configure(state='normal')
        self.withdraw()
        SistemaGestion(self, username, data.role, self.services, self.runner)

    def _failed(self, error):
        """error: fallo de login; deja corregir credenciales o reintentar."""
        self.status.configure(text=str(error))
        self.button.configure(state='normal')

    def mostrar_recuperacion(self):
        """Informa el procedimiento actual; no existe recuperación automática."""
        from tkinter import messagebox
        messagebox.showinfo('Recuperar cuenta', 'Contacte al administrador para restablecer su cuenta.', parent=self)

    def cerrar_aplicacion(self):
        """Cancela tareas y cierra HTTP antes de destruir la ventana principal."""
        if self._closing:
            return
        self._closing = True
        self.services.api.set_token(None)
        self.button.configure(state='disabled')
        self.status.configure(text='Cerrando…')
        self.runner.close(self.services.api.aclose(), self.destroy)
```

### `frontend/src/ui/sistema_gestion.py`

Ventana de navegación; delega cada módulo a su propia vista.

SHA-256: `613f5fe5c7039084d3688625da8fc4a7ae765f00fb0fdaf1bb810c1f49005017`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `SistemaGestion` | 12?65 | Compone navegación por rol con servicios y runner compartidos. |
| `SistemaGestion.__init__` | 15?44 | master: login; usuario_actual: cuenta; rol_actual: rol API; services/runner: dependencias. |
| `SistemaGestion.show_view` | 46?51 | view_class: módulo a mostrar; destruye la vista anterior y sus modales. |
| `SistemaGestion.cerrar_sesion` | 53?58 | Borra el JWT local y vuelve a login; no revoca tokens en backend. |
| `SistemaGestion.destroy` | 60?65 | Cancela tareas de la vista antes de cerrar la ventana. |

**C?digo de referencia, no ejecutable**

```python
"""Ventana de navegación; delega cada módulo a su propia vista."""
from tkinter import messagebox
import ttkbootstrap as ttk
from ui.views.clients import ClientsView
from ui.views.equipment import EquipmentView
from ui.views.users import UsersView
from ui.views.dashboard import DashboardView

ROLE_LABELS = {'admin': 'Administrador', 'recepcion': 'Recepción', 'tecnico': 'Técnico'}


class SistemaGestion(ttk.Toplevel):
    """Compone navegación por rol con servicios y runner compartidos."""

    def __init__(self, master, usuario_actual, rol_actual, services, runner):
        """master: login; usuario_actual: cuenta; rol_actual: rol API; services/runner: dependencias."""
        super().__init__(master)
        self.services, self.runner, self.role = services, runner, rol_actual
        self.view = None
        self.title('Fixify — Gestión de Equipos Tecnológicos')
        self.geometry('1100x700')
        self.minsize(900, 600)
        self.protocol('WM_DELETE_WINDOW', master.cerrar_aplicacion)
        self.columnconfigure(1, weight=1)
        self.rowconfigure(0, weight=1)
        sidebar = ttk.Frame(self, padding=15, bootstyle='secondary')
        sidebar.grid(row=0, column=0, sticky='nsew')
        ttk.Label(sidebar, text='FIXIFY', font=('Arial', 18, 'bold'), bootstyle='inverse-secondary').pack(pady=15)
        ttk.Label(sidebar, text=f'Usuario: {usuario_actual}\nRol: {ROLE_LABELS.get(self.role, self.role)}',
                  bootstyle='inverse-secondary').pack(pady=(0, 20))
        views = [('Dashboard', DashboardView)]
        if self.role in ('admin', 'recepcion'):
            views.extend([('Equipos', EquipmentView), ('Clientes', ClientsView)])
        if self.role == 'admin':
            views.append(('Usuarios', UsersView))
        for label, view_class in views:
            ttk.Button(sidebar, text=label, bootstyle='secondary',
                       command=lambda cls=view_class: self.show_view(cls)).pack(fill='x', pady=5)
        ttk.Button(sidebar, text='Cerrar sesión', bootstyle='danger',
                   command=self.cerrar_sesion).pack(side='bottom', fill='x', pady=10)
        self.content = ttk.Frame(self, padding=25)
        self.content.grid(row=0, column=1, sticky='nsew')
        self.place_window_center()
        self.show_view(DashboardView)

    def show_view(self, view_class):
        """view_class: módulo a mostrar; destruye la vista anterior y sus modales."""
        if self.view is not None:
            self.view.destroy()
        self.view = view_class(self.content, self.services, self.runner, self.role)
        self.view.pack(fill='both', expand=True)

    def cerrar_sesion(self):
        """Borra el JWT local y vuelve a login; no revoca tokens en backend."""
        if messagebox.askyesno('Cerrar sesión', '¿Desea salir del sistema?', parent=self):
            self.services.api.set_token(None)
            self.destroy()
            self.master.deiconify()

    def destroy(self):
        """Cancela tareas de la vista antes de cerrar la ventana."""
        if self.view is not None:
            self.view.destroy()
            self.view = None
        super().destroy()
```

### `frontend/src/ui/views/__init__.py`

Vistas separadas por módulo de negocio.

SHA-256: `c4c9f867f92c82bcb95af87e45acdb69c74d365bff62b3fb9992717f7b525628`.


**C?digo de referencia, no ejecutable**

```python
"""Vistas separadas por módulo de negocio."""
```

### `frontend/src/ui/views/clients.py`

Formulario y tabla del módulo de clientes.

SHA-256: `61bb289c975fb5f8f2eb7db5e82a9c704a3f20d3e3f8b81cf5d038d4849c97e0`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `ClientsView` | 6?24 | Configura clientes reutilizando la vista común de colección. |
| `ClientsView.fetch` | 14?16 | Consulta clientes desde su service. |
| `ClientsView.save` | 18?20 | values: nombre, teléfono y datos opcionales capturados. |
| `ClientsView.row` | 22?24 | record: cliente API; convierte campos opcionales vacíos a guion. |

**C?digo de referencia, no ejecutable**

```python
"""Formulario y tabla del módulo de clientes."""
from ui.components import CollectionView, Field
from models.clients import Cliente


class ClientsView(CollectionView):
    """Configura clientes reutilizando la vista común de colección."""
    title = 'Directorio de Clientes'
    columns = ('ID', 'Nombre', 'Teléfono', 'Correo', 'Dirección')
    fields = (Field('nombre', 'Nombre completo'), Field('telefono', 'Teléfono'),
              Field('correo', 'Correo (opcional)', required=False),
              Field('direccion', 'Dirección (opcional)', required=False))

    async def fetch(self):
        """Consulta clientes desde su service."""
        return await self.services.clients.listar()

    async def save(self, values):
        """values: nombre, teléfono y datos opcionales capturados."""
        return await self.services.clients.crear(**values)

    def row(self, record: Cliente):
        """record: cliente API; convierte campos opcionales vacíos a guion."""
        return (record.id, record.nombre, record.telefono, record.correo or '-', record.direccion or '-')
```

### `frontend/src/ui/views/dashboard.py`

Dashboard con consultas concurrentes y métricas limitadas por rol.

SHA-256: `61bfca4f768af6337037b4b8369cdc85327b91fccbc6cef7cf01053f4e25f450`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `DashboardView` | 7?58 | Muestra conteos sin confundir fallos de red con cero registros. |
| `DashboardView.__init__` | 10?35 | Recibe dependencias comunes y solicita solo colecciones autorizadas. |
| `DashboardView._fetch` | 37?39 | Consulta métricas en paralelo; conserva errores independientes. |
| `DashboardView._loaded` | 41?54 | results: una lista o error por métrica; muestra fallos sin ocultarlos. |
| `DashboardView._failed` | 56?58 | error: fallo general del dashboard; mantiene la ventana utilizable. |

**C?digo de referencia, no ejecutable**

```python
"""Dashboard con consultas concurrentes y métricas limitadas por rol."""
import asyncio
import ttkbootstrap as ttk
from ui.components import AsyncFrame, create_table


class DashboardView(AsyncFrame):
    """Muestra conteos sin confundir fallos de red con cero registros."""

    def __init__(self, master, services, runner, role):
        """Recibe dependencias comunes y solicita solo colecciones autorizadas."""
        super().__init__(master, services, runner, role)
        ttk.Label(self, text='Dashboard General', font=('Arial', 18, 'bold')).pack(anchor='w', pady=(0, 20))
        self.status = ttk.Label(self, text='Cargando…', wraplength=750)
        self.status.pack(fill='x', pady=10)
        cards = ttk.Frame(self)
        cards.pack(fill='x', pady=10)
        self.metrics = {}
        self.queries = []
        if role in ('admin', 'recepcion'):
            self.queries = [('Equipos', services.equipment.listar), ('Clientes', services.clients.listar)]
        if role == 'admin':
            self.queries.append(('Usuarios', services.users.listar))
        for index, (label, _) in enumerate(self.queries):
            card = ttk.Labelframe(cards, text=label, padding=15)
            card.grid(row=0, column=index, sticky='nsew', padx=5)
            cards.columnconfigure(index, weight=1)
            metric = ttk.Label(card, text='…', font=('Arial', 22, 'bold'))
            metric.pack()
            self.metrics[label] = metric
        if not self.queries:
            self.status.configure(text='Sesión de técnico activa. El módulo de órdenes asignadas está pendiente.')
            return
        self.table = create_table(self, ('Folio', 'Tipo', 'Marca', 'Modelo', 'Problema', 'ID Cliente'), height=8)
        self.runner.submit(self._fetch(), self, self._loaded, self._failed)

    async def _fetch(self):
        """Consulta métricas en paralelo; conserva errores independientes."""
        return await asyncio.gather(*(fetch() for _, fetch in self.queries), return_exceptions=True)

    def _loaded(self, results):
        """results: una lista o error por métrica; muestra fallos sin ocultarlos."""
        errors = []
        for (label, _), result in zip(self.queries, results):
            if isinstance(result, Exception):
                self.metrics[label].configure(text='No disponible')
                errors.append(f'{label}: {result}')
                continue
            self.metrics[label].configure(text=str(len(result)))
            if label == 'Equipos':
                for equipment in sorted(result, key=lambda eq: eq.id, reverse=True)[:8]:
                    self.table.insert('', 'end', values=(f"EQ-{equipment.id:03d}", equipment.tipo,
                        equipment.marca, equipment.modelo, equipment.problema_reportado, equipment.client_id))
        self.status.configure(text='\n'.join(errors) or 'Datos actualizados. Equipos ordenados por ID descendente.')

    def _failed(self, error):
        """error: fallo general del dashboard; mantiene la ventana utilizable."""
        self.status.configure(text=str(error))
```

### `frontend/src/ui/views/equipment.py`

Vista de equipos con selección asíncrona de clientes existentes.

SHA-256: `8b38400126b79f6e2d316e5ea93fee9edcdbb6d7c435d2a1da0793e0d512be2f`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `EquipmentView` | 7?52 | Reutiliza la tabla y prepara el selector de propietario antes del alta. |
| `EquipmentView.fetch` | 12?14 | Consulta los equipos disponibles mediante su service. |
| `EquipmentView.row` | 16?19 | record: equipo API; formatea su identificador visual. |
| `EquipmentView.open_form` | 21?28 | Carga propietarios sin detener Tk; evita consultas duplicadas. |
| `EquipmentView._clients_failed` | 30?33 | error: fallo de consulta; restaura el botón de alta. |
| `EquipmentView._clients_ready` | 35?52 | clients: lista API; crea etiquetas legibles asociadas a IDs. |
| `EquipmentView._clients_ready.save` | 47?49 | values: formulario; traduce la etiqueta elegida a ID de cliente. |

**C?digo de referencia, no ejecutable**

```python
"""Vista de equipos con selección asíncrona de clientes existentes."""
from ui.components import CollectionView, Field, FormDialog
from models.equipment import Equipo
from models.clients import Cliente


class EquipmentView(CollectionView):
    """Reutiliza la tabla y prepara el selector de propietario antes del alta."""
    title = 'Gestión de Equipos'
    columns = ('Folio', 'Tipo', 'Marca', 'Modelo', 'Nº Serie', 'Problema', 'ID Cliente')

    async def fetch(self):
        """Consulta los equipos disponibles mediante su service."""
        return await self.services.equipment.listar()

    def row(self, record: Equipo):
        """record: equipo API; formatea su identificador visual."""
        return (f"EQ-{record.id:03d}", record.tipo, record.marca, record.modelo,
                record.numero_serie or '-', record.problema_reportado, record.client_id)

    def open_form(self):
        """Carga propietarios sin detener Tk; evita consultas duplicadas."""
        if self._dialog is not None and self._dialog.winfo_exists():
            self._dialog.lift()
            return
        self.create.configure(state='disabled')
        self.status.configure(text='Cargando clientes…')
        self.runner.submit(self.services.clients.listar(), self, self._clients_ready, self._clients_failed)

    def _clients_failed(self, error):
        """error: fallo de consulta; restaura el botón de alta."""
        self.create.configure(state='normal')
        self.status.configure(text=str(error))

    def _clients_ready(self, clients: list[Cliente]):
        """clients: lista API; crea etiquetas legibles asociadas a IDs."""
        self.create.configure(state='normal')
        if not clients:
            self.status.configure(text='Registre al menos un cliente antes de ingresar un equipo.')
            return
        owners = {f"[{c.id}] {c.nombre} ({c.telefono})": c.id for c in clients}
        fields = (Field('client_id', 'Cliente', choices=tuple(owners)),
                  Field('tipo', 'Tipo de equipo'), Field('marca', 'Marca'),
                  Field('modelo', 'Modelo'), Field('numero_serie', 'Serie (opcional)', required=False),
                  Field('problema_reportado', 'Problema reportado'))

        async def save(values):
            """values: formulario; traduce la etiqueta elegida a ID de cliente."""
            return await self.services.equipment.crear(**{**values, 'client_id': owners[values['client_id']]})

        self._dialog = FormDialog(self, 'Registrar Equipo', fields, self.runner, save, self._created)
        self.status.configure(text='Seleccione el cliente propietario en el formulario.')
```

### `frontend/src/ui/views/users.py`

Vista de administración de cuentas y roles.

SHA-256: `9eeded97348433429aa64d3fb6fcfa41d59fe75665a0bda0f7365574d025f855`.

| S?mbolo | L?neas | Prop?sito / argumentos |
|---|---|---|
| `UsersView` | 10?84 | Permite altas y cambios de acceso sobre el usuario seleccionado. |
| `UsersView.__init__` | 18?31 | Recibe dependencias comunes y agrega acciones administrativas. |
| `UsersView.fetch` | 33?35 | Consulta personal autorizado para administración. |
| `UsersView.save` | 37?39 | values: credenciales y etiqueta de rol; convierte esta al valor API. |
| `UsersView.row` | 41?43 | record: usuario API; muestra estado legible sin datos de contraseña. |
| `UsersView._selected` | 45?51 | Retorna el registro seleccionado o informa que falta selección. |
| `UsersView.change_role` | 53?57 | Solicita cambio al rol seleccionado para la cuenta elegida. |
| `UsersView.toggle_active` | 59?63 | Alterna acceso sin eliminar el historial del usuario. |
| `UsersView._update` | 65?72 | coroutine: actualización administrativa; bloquea acciones repetidas. |
| `UsersView._updated` | 74?78 | result: respuesta API; restaura acciones y recarga cuentas. |
| `UsersView._update_failed` | 80?84 | error: fallo; permite reintentar sin ocultar la causa. |

**C?digo de referencia, no ejecutable**

```python
"""Vista de administración de cuentas y roles."""
from tkinter import messagebox
import ttkbootstrap as ttk
from ui.components import CollectionView, Field
from models.users import Usuario

ROLES = {'Administrador': 'admin', 'Técnico': 'tecnico', 'Recepción': 'recepcion'}


class UsersView(CollectionView):
    """Permite altas y cambios de acceso sobre el usuario seleccionado."""
    title = 'Administración de Usuarios'
    columns = ('ID', 'Usuario', 'Rol', 'Estado')
    fields = (Field('username', 'Usuario (mín. 4 caracteres)'),
              Field('password', 'Contraseña (mín. 6 caracteres)', secret=True),
              Field('role', 'Rol', choices=tuple(ROLES)))

    def __init__(self, master, services, runner, role):
        """Recibe dependencias comunes y agrega acciones administrativas."""
        super().__init__(master, services, runner, role)
        actions = ttk.Frame(self)
        actions.pack(fill='x', pady=10)
        self.role_input = ttk.Combobox(actions, values=tuple(ROLES), state='readonly', width=18)
        self.role_input.set('Técnico')
        self.role_input.pack(side='left', padx=(0, 10))
        self.action_buttons = []
        for label, callback in (('Cambiar rol', self.change_role),
                                ('Activar / Desactivar', self.toggle_active)):
            button = ttk.Button(actions, text=label, command=callback)
            button.pack(side='left', padx=5)
            self.action_buttons.append(button)

    async def fetch(self):
        """Consulta personal autorizado para administración."""
        return await self.services.users.listar()

    async def save(self, values):
        """values: credenciales y etiqueta de rol; convierte esta al valor API."""
        return await self.services.users.crear(**{**values, 'role': ROLES[values['role']]})

    def row(self, record: Usuario):
        """record: usuario API; muestra estado legible sin datos de contraseña."""
        return (record.id, record.username, record.role, 'Activo' if record.is_active else 'Inactivo')

    def _selected(self):
        """Retorna el registro seleccionado o informa que falta selección."""
        selected = self.table.selection()
        if not selected:
            self.status.configure(text='Seleccione un usuario en la tabla.')
            return None
        return self.records[int(selected[0])]

    def change_role(self):
        """Solicita cambio al rol seleccionado para la cuenta elegida."""
        user = self._selected()
        if user and messagebox.askyesno('Cambiar rol', '¿Cambiar el rol de esta cuenta?', parent=self):
            self._update(self.services.users.cambiar_rol(user.id, ROLES[self.role_input.get()]))

    def toggle_active(self):
        """Alterna acceso sin eliminar el historial del usuario."""
        user = self._selected()
        if user and messagebox.askyesno('Cambiar acceso', '¿Cambiar el acceso de esta cuenta?', parent=self):
            self._update(self.services.users.activar(user.id, not user.is_active))

    def _update(self, coroutine):
        """coroutine: actualización administrativa; bloquea acciones repetidas."""
        if self._load_future is not None:
            self._load_future.cancel()
        for button in [self.refresh, self.create, *self.action_buttons]:
            button.configure(state='disabled')
        self.status.configure(text='Actualizando cuenta…')
        self.runner.submit(coroutine, self, self._updated, self._update_failed)

    def _updated(self, result):
        """result: respuesta API; restaura acciones y recarga cuentas."""
        for button in [self.create, *self.action_buttons]:
            button.configure(state='normal')
        self.load()

    def _update_failed(self, error):
        """error: fallo; permite reintentar sin ocultar la causa."""
        for button in [self.create, *self.action_buttons]:
            button.configure(state='normal')
        self._failed(error)
```

## 16. Comprobaciones y pr?ximos pasos

Comandos desde la ra?z:

```powershell
python -m unittest discover -s frontend/tests -v
python -m unittest discover -s backend/tests -v
python frontend/src/app/main.py
```

Las pruebas de frontend verifican 15 escenarios de contratos, modelos, errores, filas de vistas, concurrencia y ciclo de cierre sin abrir Tk. Las pruebas de equipos verifican 5 escenarios: alta/listado con columnas f?sicas del diagrama, validaci?n y cliente inexistente, serie opcional/orden estable, permisos y consumo del router real desde EquipmentService con ASGITransport. Utilizan SQLite temporal: comprueban mapeo/contrato, no la conectividad ni todas las particularidades de PostgreSQL.

Pendiente: aceptaci?n visual y ejecuci?n contra esquema real de Supabase. El diagrama define nombres, no evidencia que todas las columnas de los otros m?dulos est?n sincronizadas. No ejecutar el SQL destructivo del repositorio para comprobar lectura.

### Referencias

- C?digo vigente: `frontend/src/` y `backend/src/modules/equipment/`.
- Diagrama proporcionado: `D:/Descargas/Diagramas.pdf`, p?gina 1; referencia de columnas f?sicas.
- [HTTPX: AsyncClient y cierre](https://www.python-httpx.org/async/).
- [HTTPX: par?metros, cuerpos y respuestas](https://www.python-httpx.org/quickstart/).
- [HTTPX: timeout](https://www.python-httpx.org/advanced/timeouts/).
- [Python: coroutines, gather y coordinaci?n entre hilos](https://docs.python.org/3/library/asyncio-task.html).

La descripci?n del flujo y sus l?mites corresponde al c?digo local. Las referencias t?cnicas explican APIs de bibliotecas; no implican que Fixify implemente todas sus capacidades.

## Actualizaci?n: alineaci?n SQL y prueba real en Supabase (02-Oct-2026)

El backend ahora mapea las diez tablas del SQL local, incluidas usuarios/roles. Se renombr? equipos.problema_reportado a falla_reportada en Supabase conservando datos; users y la columna adicional direccion se conservan remotamente, pero no forman parte del ORM del SQL base. No hay create_all durante el arranque.

Se cre? fixify_prueba (ID 2, recepci?n) y se verific? login real y serializaci?n de clientes/equipos contra Supabase. Su contrase?a est? en backend/usuario_prueba.local.json, excluido de Git. Pasan 7 pruebas backend y 15 frontend.

Limitaciones del esquema original: no existe is_active; activar/desactivar retorna 409 hasta acordar una extensi?n. Direcci?n no pertenece al SQL: el backend devuelve None y rechaza alta con direcci?n no vac?a. No se migraron cuentas de users autom?ticamente. V?ase [detalle de alineaci?n](ALINEACION_BASE_DATOS.md). Esta actualizaci?n sustituye las afirmaciones anteriores de falta de validaci?n remota para login/clientes/equipos; la aceptaci?n visual permanece pendiente.


## Actualizaci?n: estado persistido de usuarios

El usuario a?adi? usuarios.is_activate en Supabase. El backend utiliza Column('is_activate', Boolean, ...) bajo el atributo is_active, manteniendo UserResponse y el frontend. Activar/desactivar vuelve a persistir el estado; login rechaza cuentas inactivas (403). El SQL local incorpora BOOLEAN NOT NULL DEFAULT TRUE; actualizar el diagrama para reflejar esta extensi?n.

Las pruebas locales verifican desactivar, rechazo de login, conservaci?n del rol, reactivar y nuevo login exitoso. No se ejecutaron cambios remotos ni se comprob? la configuraci?n de defaults/nullabilidad de la columna existente. Esta actualizaci?n sustituye la limitaci?n anterior de activaci?n no soportada. La autorizaci?n de JWT ya emitidos mantiene su comportamiento previo.


## Correcci?n de error 500: nombre del estado de cuenta

El traceback de PostgreSQL confirm? que la columna existente es usuarios.is_active, no is_activate. Se corrigi? el mapeo ORM y el SQL local para usar is_active. El contrato del frontend no cambia. Ocho pruebas backend pasan, incluida una regresi?n que comprueba el nombre f?sico de la columna. Esta correcci?n sustituye las referencias anteriores a is_activate; no requiere renombrar la columna en Supabase.
